# LAMBADA contexts for cross-family stitching
Validation and test are passages whose last word is the label. Train is the full novels, so a training example has to be cut into a passage of that same shape before it can be labeled the same way.

Run on CPU to sample and save 10,000 labeled training passages. In Colab, enable the HF_TOKEN secret for the gated Llama tokenizer and mount the shared Drive before saving. The manifest cell prints the saved file path. Use activation.ipynb for GPU extraction.

## Install


In [1]:
%pip install -q \
    "transformers>=5,<6" \
    "huggingface_hub>=1.32,<2" \
    "datasets>=3,<5" \
    "accelerate>=1.0" \
    safetensors \
    tqdm \
    "nltk>=3.9.1,<4" \
    numpy \
    matplotlib


## Imports


In [2]:
import hashlib
import json
import math
import os
import random
import re
from collections import Counter
from pathlib import Path

from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import AutoTokenizer
from tqdm.auto import tqdm


## Configuration


In [3]:
SEED = 42
N_TRAIN = 10_000
MAX_TOKENS = 256

DATA = "cimec/lambada"
MODELS = {"llama": "meta-llama/Llama-3.2-3B", "qwen": "Qwen/Qwen2.5-3B"}
SPLITS = ("train", "validation", "test")

## Log in to Hugging Face
In Colab, add a secret named `HF_TOKEN` in the Secrets panel and turn on notebook access. Accept the Llama license at [meta-llama/Llama-3.2-3B](https://huggingface.co/meta-llama/Llama-3.2-3B) before running this cell. Outside Colab, set the `HF_TOKEN` environment variable before starting the kernel.

In [4]:
import os

from huggingface_hub import hf_hub_download, login, whoami
from huggingface_hub.utils import GatedRepoError

token = os.environ.get("HF_TOKEN")
if token is None:
    from google.colab import userdata
    try:
        token = userdata.get("HF_TOKEN")
    except userdata.SecretNotFoundError as exc:
        raise RuntimeError("Secret 'HF_TOKEN' not found. Add it in the Colab Secrets sidebar.") from exc
    except userdata.NotebookAccessError as exc:
        raise RuntimeError("Notebook access disabled. Toggle on 'Notebook access' for HF_TOKEN.") from exc

login(token=token)
print(f"Authenticated as: {whoami()['name']}")
try:
    hf_hub_download(repo_id=MODELS["llama"], filename="config.json")
except GatedRepoError as exc:
    raise RuntimeError(
        f"Access denied for {MODELS['llama']}. Accept the license on its Hugging Face page."
    ) from exc
print(f"Access confirmed for {MODELS['llama']}")

Authenticated as: edlose


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

Access confirmed for meta-llama/Llama-3.2-3B


## Load every split


In [5]:
api = HfApi()
data_rev = api.dataset_info(DATA).sha
dataset = load_dataset(DATA, revision=data_rev)
{name: len(dataset[name]) for name in SPLITS}


README.md:   0%|          | 0.00/7.32k [00:00<?, ?B/s]

plain_text/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  269MB            

plain_text/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

plain_text/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  281MB            

plain_text/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.14MB            

... (3 outputs left out)

Generating train split:   0%|          | 0/2662 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5153 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4869 [00:00<?, ? examples/s]

{'train': 2662, 'validation': 4869, 'test': 5153}

## Passage helpers


In [6]:
WORD = re.compile(r"\S+")
LETTER_WORD = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")


def quantiles(values):
    xs = sorted(values)
    n = len(xs)
    out = {"n": n, "min": xs[0], "max": xs[-1], "mean": round(sum(xs) / n, 1)}
    for p in (5, 25, 50, 75, 95):
        out[f"p{p}"] = xs[min(n - 1, round(p / 100 * (n - 1)))]
    return out


def split_last_word(text):
    """Prefix, continuation, and word count for a passage whose label is its last word."""
    spans = list(WORD.finditer(text))
    if len(spans) < 2 or spans[-1].end() != len(text):
        return None
    if LETTER_WORD.fullmatch(spans[-1].group()) is None:
        return None
    end = spans[-2].end()
    return text[:end], text[end:], len(spans)


def histogram(counts, width=5, unit=20):
    buckets = Counter((n // width) * width for n in counts)
    for start in range(min(buckets), max(buckets) + 1, width):
        bar = "#" * (buckets[start] // unit)
        print(f"{start:4d}-{start + width - 1:<4d} {buckets[start]:5d} {bar}")


def passage_report(split):
    separators = Counter()
    word_counts, context_words = [], []
    dropped = 0
    for text in dataset[split]["text"]:
        parts = split_last_word(text)
        if parts is None:
            dropped += 1
            continue
        _prefix, continuation, n_words = parts
        target = continuation.split()[-1]
        separators[continuation[:-len(target)]] += 1
        word_counts.append(n_words)
        context_words.append(n_words - 1)
    print(f"{split}: kept {len(word_counts):,}, dropped {dropped:,}")
    print("  passage words", quantiles(word_counts))
    print("  context words", quantiles(context_words))
    print("  separator before target", separators.most_common(5))
    return word_counts


## Inventory
Train rows are novels and carry a `domain`. Validation and test rows are passages, and `domain` is empty on both.


In [7]:
print(dataset["train"].features)
for split in SPLITS:
    domains = dataset[split]["domain"]
    filled = sum(domain is not None for domain in domains)
    if split == "train":
        print(f"train: {len(dataset[split]):,} novels, domain filled on {filled:,}")
        continue
    texts = dataset[split]["text"]
    print(f"\n{split}: {len(texts):,} rows, domain filled on {filled:,}")
    print("  chars", quantiles(len(text) for text in texts))
    print("  words", quantiles(len(WORD.findall(text)) for text in texts))
print("\ntrain domains")
print(Counter(dataset["train"]["domain"]).most_common())
print("\ntrain excerpt:", repr(dataset["train"][0]["text"][:300]))


{'text': Value('string'), 'domain': Value('string')}
train: 2,662 novels, domain filled on 2,662

validation: 4,869 rows, domain filled on 0
  chars {'n': 4869, 'min': 217, 'max': 953, 'mean': 341.7, 'p5': 264, 'p25': 298, 'p50': 332, 'p75': 373, 'p95': 453}
  words {'n': 4869, 'min': 59, 'max': 170, 'mean': 74.1, 'p5': 62, 'p25': 67, 'p50': 72, 'p75': 79, 'p95': 94}

test: 5,153 rows, domain filled on 0
  chars {'n': 5153, 'min': 212, 'max': 986, 'mean': 339.6, 'p5': 262, 'p25': 297, 'p50': 329, 'p75': 370, 'p95': 451}
  words {'n': 5153, 'min': 58, 'max': 203, 'mean': 73.7, 'p5': 61, 'p25': 66, 'p50': 72, 'p75': 79, 'p95': 93}

train domains
[('Romance', 600), ('Fantasy', 581), ('New_Adult', 218), ('Mystery', 183), ('Science_fiction', 176), ('Adventure', 141), ('Teen', 132), ('Horror', 115), ('Thriller', 108), ('Literature', 91), ('Humor', 73), ('Other', 71), ('Historical', 69), ('Vampires', 53), ('Young_Adult', 44), ('Themes', 7)]

train excerpt: 'prism story of ci prism -lrb- story

## How a validation passage is labeled
The label is the last whitespace-delimited word. The continuation keeps the separator in front of that word, and the prefix is everything before it. Test uses the same convention. A training window has to meet these checks before it is the same kind of example.


In [8]:
length_counts = {}
for split in ("validation", "test"):
    length_counts[split] = passage_report(split)

print("\nvalidation examples")
for text in dataset["validation"]["text"][:3]:
    prefix, continuation, n_words = split_last_word(text)
    print(n_words, "words |", repr(prefix), "→", repr(continuation))
validation_lengths = length_counts["validation"]

validation: kept 4,869, dropped 0
  passage words {'n': 4869, 'min': 59, 'max': 170, 'mean': 74.1, 'p5': 62, 'p25': 67, 'p50': 72, 'p75': 79, 'p95': 94}
  context words {'n': 4869, 'min': 58, 'max': 169, 'mean': 73.1, 'p5': 61, 'p25': 66, 'p50': 71, 'p75': 78, 'p95': 93}
  separator before target [(' ', 4869)]
test: kept 5,153, dropped 0
  passage words {'n': 5153, 'min': 58, 'max': 203, 'mean': 73.7, 'p5': 61, 'p25': 66, 'p50': 72, 'p75': 79, 'p95': 93}
  context words {'n': 5153, 'min': 57, 'max': 202, 'mean': 72.7, 'p5': 60, 'p25': 65, 'p50': 71, 'p75': 78, 'p95': 92}
  separator before target [(' ', 5153)]

validation examples
101 words | 'her pay for the evening was almost double that of the wait staff and although that might not seem like a lot to some people , it was a small fortune to claire . after loading her final tray for a server , claire went to the restroom to freshen up and begin preparations for being loaded into the cake . pam had a couple of young men from college wh

## What to copy from validation when cutting a novel
Validation and test passages run from about 60 to 100 words and sit near 70. They do not start on a sentence boundary. Every gap between words is a single space, the passage ends on the target, and that target is an alphabetic word.

Draw window lengths from the validation word counts. Join the sampled training words with single spaces before tokenization.


In [9]:
print("validation passage length")
histogram(validation_lengths, width=5, unit=30)
print("\ntest passage length")
histogram(length_counts["test"], width=5, unit=30)


validation passage length
  55-59      35 #
  60-64     720 ########################
  65-69    1171 #######################################
  70-74     999 #################################
  75-79     778 #########################
  80-84     450 ###############
  85-89     297 #########
  90-94     191 ######
  95-99      96 ###
 100-104     61 ##
 105-109     27 
 110-114     16 
 115-119      9 
 120-124      5 
 125-129      3 
 130-134      4 
 135-139      4 
 140-144      1 
 145-149      0 
 150-154      0 
 155-159      1 
 160-164      0 
 165-169      0 
 170-174      1 

test passage length
  55-59      27 
  60-64     836 ###########################
  65-69    1242 #########################################
  70-74    1094 ####################################
  75-79     764 #########################
  80-84     478 ###############
  85-89     318 ##########
  90-94     178 #####
  95-99      82 ##
 100-104     57 #
 105-109     26 
 110-114     23 
 115-119     10 
 120-

## Tokenizers
Both models label every kept passage. A row is dropped if either tokenizer splits the prefix and the continuation boundary apart, or if the prefix is longer than `MAX_TOKENS`.


In [10]:
revisions = {name: api.model_info(repo).sha for name, repo in MODELS.items()}
tokenizers = {
    name: AutoTokenizer.from_pretrained(repo, revision=revisions[name])
    for name, repo in MODELS.items()
}
for tok in tokenizers.values():
    tok.padding_side = "right"
    tok.pad_token = tok.eos_token


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## Attach both vocabularies to one passage


In [11]:
def annotate_tokens(row):
    prefix = row["prefix"]
    full = prefix + row["continuation"]
    for name, tok in tokenizers.items():
        ids = tok.encode(prefix)
        complete = tok.encode(full)
        if len(ids) > MAX_TOKENS or complete[:len(ids)] != ids or len(complete) <= len(ids):
            return None
        row[name + "_length"] = len(ids)
        row[name + "_target_id"] = complete[len(ids)]
        row[name + "_target_length"] = len(complete) - len(ids)
    return row


def label_passage(text, example_id, split):
    parts = split_last_word(text)
    if parts is None:
        return None
    prefix, continuation, n_words = parts
    return annotate_tokens(dict(
        id=example_id, split=split, n_words=n_words,
        prefix=prefix, continuation=continuation,
    ))


def passages_to_rows(split):
    rows, dropped = [], 0
    for index, text in enumerate(dataset[split]["text"]):
        row = label_passage(text, f"{split}/{index}", split)
        if row is None:
            dropped += 1
            continue
        rows.append(row)
    print(f"{split}: {len(rows):,} labeled, {dropped:,} dropped")
    return rows


## Validation and test rows


In [12]:
rows_by_split = {}
for split in ("validation", "test"):
    rows_by_split[split] = passages_to_rows(split)
    row = rows_by_split[split][0]
    print(repr(row["prefix"][-100:]), "→", repr(row["continuation"]))
    print({name: tok.decode([row[name + "_target_id"]]) for name, tok in tokenizers.items()})


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


validation: 4,869 labeled, 0 dropped
'. brian and max were a lot of fun and always made her laugh as they hoisted her up to the top of the' → ' cake'
{'llama': ' cake', 'qwen': ' cake'}
test: 5,153 labeled, 0 dropped
"sort of sign . but as i stare at harlin , his mouth curved in a confident grin , i do n't care about" → ' signs'
{'llama': ' signs', 'qwen': ' signs'}


## Sample training passages with noun targets
Draw passage lengths from validation and keep training endpoints tagged as common or proper nouns (singular or plural). The English POS tagger sees the passage and up to eight following words from the novel to help disambiguate the target. Only the passage prefix goes to Llama and Qwen. Each accepted training row records its target tag.

The sampler probes random character offsets and accepts word starts, then reads only the nearby context. Sampling runs in one process. The first pass spreads examples across novels. Further passes revisit productive novels until the target count is reached or a pass yields no new examples. Every training target must also occur earlier as a whitespace-delimited word in the same prefix (case-insensitive).


In [13]:
from itertools import islice
import nltk
from nltk.tag import PerceptronTagger

nltk.download("averaged_perceptron_tagger_eng", quiet=True, raise_on_error=True)
pos_tagger = PerceptronTagger(lang="eng")
NOUN_TAGS = {"NN", "NNS", "NNP", "NNPS"}


def target_in_prefix(prefix_words, target):
    target = target.casefold()
    return any(word.casefold() == target for word in prefix_words)


# Accept only offsets at the beginning of a word. Each word has one such
# offset, so long words receive the same sampling probability as short ones.
def sample_novel(novel_id, rng, quota, seen):
    text = dataset["train"][novel_id]["text"]
    rows, tried = [], set()
    if not text:
        return rows
    for _ in range(1200 * quota):
        pos = rng.randrange(len(text))
        if pos == 0 or not text[pos - 1].isspace() or pos in tried:
            continue
        target = LETTER_WORD.match(text, pos)
        if target is None or (target.end() < len(text) and not text[target.end()].isspace()):
            continue
        tried.add(pos)
        n_words = rng.choice(validation_lengths)
        # Expand only if this local slice might begin inside the desired window.
        width = 8 * n_words
        while True:
            left = max(0, pos - width)
            words = list(WORD.finditer(text, left, pos))
            if left == 0 or len(words) >= n_words:
                break
            width *= 2
        if len(words) < n_words - 1:
            continue
        start = words[-(n_words - 1)].start()
        prefix_end = words[-1].end()
        passage = text[start:target.end()]
        words_in_passage = passage.split()
        prefix_words = words_in_passage[:-1]
        if not target_in_prefix(prefix_words, words_in_passage[-1]):
            continue
        prefix = " ".join(prefix_words)
        if prefix in seen:
            continue
        following = [m.group() for m in islice(WORD.finditer(text, target.end()), 8)]
        tag = pos_tagger.tag(words_in_passage + following)[-1 - len(following)][1]
        if tag not in NOUN_TAGS:
            continue
        row = annotate_tokens(dict(id=f"train/{novel_id}/{start}/{prefix_end}", split="train",
                                   n_words=n_words, prefix=prefix, continuation=" " + target.group(),
                                   source_target_end=target.end()))
        if row is None:
            continue
        row["target_pos"] = tag
        seen.add(prefix)
        rows.append(row)
        if len(rows) == quota:
            break
    return rows


def sample_training_rows():
    rng = random.Random(SEED)
    order = list(range(len(dataset["train"])))
    rng.shuffle(order)
    quota = max(1, math.ceil(N_TRAIN / len(order)))
    rows, seen = [], set()
    with tqdm(total=N_TRAIN, desc="training passages") as progress:
        while order:
            productive = []
            for novel_id in order:
                batch = sample_novel(novel_id, random.Random(rng.getrandbits(64)),
                                     min(quota, N_TRAIN - len(rows)), seen)
                if batch:
                    productive.append(novel_id)
                    rows.extend(batch)
                    progress.update(len(batch))
                if len(rows) == N_TRAIN:
                    return rows
            order = productive
            rng.shuffle(order)
    raise ValueError(f"Only {len(rows)} unique training passages found.")


rows_by_split["train"] = sample_training_rows()
print(f"{len(rows_by_split['train']):,} training passages")
print(Counter(row["target_pos"] for row in rows_by_split["train"]))


training passages:   0%|          | 0/10000 [00:00<?, ?it/s]

10,000 training passages
Counter({'NN': 8917, 'NNS': 1075, 'NNP': 7, 'NNPS': 1})


## Check the training sample against validation and test


In [14]:
def preview(split, k=3):
    print(f"\n{split} words", quantiles(row["n_words"] for row in rows_by_split[split]))
    for row in rows_by_split[split][:k]:
        print(row["n_words"], repr(row["prefix"]), "→", repr(row["continuation"]))


for split in SPLITS:
    preview(split)



train words {'n': 10000, 'min': 59, 'max': 170, 'mean': 75.0, 'p5': 62, 'p25': 67, 'p50': 73, 'p75': 80, 'p95': 95}
68 '. once he disturbed a large mehrr-koh , which blundered screeching from tree to tree before flapping heavily away , the scarlet and purple of its back and wing feathers contrasting sharply with the rich yellow hues on its breast . after walking for a long time , up ahead to his right , kreh-ursh spied one : a taat-eh tree , a single one . his' → ' tree'
77 ', watery slope , his canoe spinning helplessly . the sea was gray-white , the light deep lilac . the tunnel spat them from its maw , and sky returned . spray crashed , bursting in geysers all around , but the storm had passed over . while the sea still bucked , rough and angry , overhead clouds transmuted into a scarlet twilight . that was when he saw it behind him -- an open' → ' maw'
61 "board toward him , opening up a channel through the ooze . though kyle scrabbled wildly , his oily hands slipped , unable to g

## Mount Google Drive
Add a shortcut from the shared `coms-4705-project` folder into My Drive: open the folder, then **Organize > Add shortcut > My Drive**. The following cell writes the sampled training passages there. Activation files are not written in this run.


In [15]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/coms-4705-project")
if not DRIVE_PROJECT_DIR.is_dir():
    raise RuntimeError("Add a My Drive shortcut to the shared coms-4705-project folder.")
print(DRIVE_PROJECT_DIR)


Mounted at /content/drive
/content/drive/MyDrive/coms-4705-project


## Save the training passages
Each JSON line is one labeled training example. The files land in `datasets/lambada/train` on the shared Drive, with the dataset revision, tokenizer revisions, and checksum beside them.


In [16]:
def write_training_set(rows):
    if len(rows) != N_TRAIN or any(row["target_pos"] not in NOUN_TAGS for row in rows):
        raise ValueError("Expected a complete training set with a noun tag on every target.")
    if any(not target_in_prefix(r["prefix"].split(), r["continuation"].strip()) for r in rows):
        raise ValueError("Every training target must occur in its own prefix.")
    manifest = "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows)
    provenance = dict(
        dataset=DATA, split="train", dataset_revision=data_rev, models=MODELS,
        revisions=revisions, seed=SEED, max_tokens=MAX_TOKENS, n_train=N_TRAIN,
        target_pos_tags=sorted(NOUN_TAGS), tagger="averaged_perceptron_tagger_eng",
        target_in_prefix="case-insensitive whitespace-delimited word",
        whitespace="single spaces", sampling="balanced first pass with productive-novel refill",
        nltk_version=nltk.__version__,
        manifest_sha256=hashlib.sha256(manifest.encode()).hexdigest(),
    )
    fingerprint = hashlib.sha256(json.dumps(provenance, sort_keys=True).encode()).hexdigest()
    run_dir = DRIVE_PROJECT_DIR / "datasets" / "lambada" / "train" / fingerprint[:16]
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "examples.jsonl").write_text(manifest, encoding="utf-8")
    (run_dir / "provenance.json").write_text(json.dumps(provenance, indent=2) + "\n", encoding="utf-8")
    print(run_dir / "examples.jsonl")
    return run_dir, fingerprint, provenance


run_dir, fingerprint, provenance = write_training_set(rows_by_split["train"])
print(f"{len(rows_by_split['train']):,} training passages saved")


/content/drive/MyDrive/coms-4705-project/datasets/lambada/train/7b6847a7f51353d0/examples.jsonl
10,000 training passages saved


## Assumptions

Source: Sample contiguous passages from LAMBADA training novels. Seed 42 makes the sampling reproducible.

Length: Draw each passage’s word count from the observed validation lengths. That count includes the target word.

Target: Use the final word. It must contain English letters, with at most one internal apostrophe, and be tagged as a common or proper noun, singular or plural.

Earlier occurrence: The target must appear as a whole word in the prefix, ignoring case. Alice matches alice; doorway does not match door.
Whitespace: Collapse spaces, tabs, and newlines to single spaces before tokenization. Keep one space before the target.

Tokenizer checks: The prefix must fit within 256 tokens for both Llama and Qwen. For each model, appending the target must preserve the prefix token IDs and add at least one token. Multi-token targets are allowed.

Uniqueness: Each normalized prefix must be unique across the training set. Target words may repeat.

Quota: Aim for 10,000 examples, taking up to four per novel per pass. Later passes revisit novels that yielded examples in the preceding pass.